# Round 3: Synthetic Comment Generation (DG / SAG / SAGII)

This notebook generates synthetic comments for r/AskDocs OPs using three distinct
prompting strategies, each designed to test how generation method affects the
realism and diversity of synthetic discourse.

**Prompting strategies:**

1. **DG (Discourse Generation)**: A single prompt generates the entire thread of *n*
   replies, matching the actual clinician/layperson distribution from the real thread.

2. **SAG (Single Advice Generation)**: *n* independent API calls, each generating a
   single response with a specified role (clinician or layperson). Each call sees only
   the OP.

3. **SAGII (SAG with Incremental Information)**: *n* sequential API calls. Each call
   generates a single response with a specified role, but also sees all previously
   generated responses in the thread so far.

**Key design choices:**
- DG uses the actual clinician/layperson ratio from each real thread
- SAG and SAGII assign roles per-query in a sequence matching the real thread's ratio
- OPs are filtered by an expanded term list (health, illness, disease, etc.) in both title and body
- Clinician/layperson counts are computed directly from comment flairs (not from pre-computed submission fields)
- 100 OPs are sampled for generation
- Checkpointing supports resumption after interruption
- Output is an Excel spreadsheet (1 tab per OP) for review

## 1. Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install google-genai tqdm openpyxl

In [1]:
import json
import os
import re
import random
import time
from getpass import getpass
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from google import genai
from google.genai import types

In [2]:
# ---------- Configuration ----------
API_KEY = getpass("Enter your Google Gemini API key: ")
MODEL_ID = "gemini-2.5-pro"

# Paths  (notebook runs from src/, so corpora is at ../output/corpora)
CORPORA_DIR = Path("../output/corpora")
SUBMISSIONS_PATH = CORPORA_DIR / "submissions_corpus.jsonl"
COMMENTS_PATH = CORPORA_DIR / "comments_corpus.jsonl"
CHECKPOINT_DIR = CORPORA_DIR / "checkpoints_round3"
OUTPUT_XLSX = CORPORA_DIR / "round3_synthetic_comments.xlsx"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Rate-limiting
DELAY_BETWEEN_CALLS = 1.0  # seconds between API calls

# Number of OPs to generate for
N_SAMPLE = 100

# Random seed for reproducibility
SEED = 42
random.seed(SEED)

print(f"Model: {MODEL_ID}")
print(f"Submissions: {SUBMISSIONS_PATH}")
print(f"Comments: {COMMENTS_PATH}")
print(f"Output spreadsheet: {OUTPUT_XLSX}")
print(f"Sample size: {N_SAMPLE} OPs")

Enter your Google Gemini API key:  ········


Model: gemini-2.5-pro
Submissions: ../output/corpora/submissions_corpus.jsonl
Comments: ../output/corpora/comments_corpus.jsonl
Output spreadsheet: ../output/corpora/round3_synthetic_comments.xlsx
Sample size: 100 OPs


In [3]:
# Initialize the Gemini client
client = genai.Client(api_key=API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


## 2. Load Data & Filter OPs

In [4]:
# Load all submissions
submissions = []
with open(SUBMISSIONS_PATH) as f:
    for line in f:
        submissions.append(json.loads(line))

print(f"Loaded {len(submissions):,} total submissions")

# Build a lookup by submission id
submissions_by_id = {s["id"]: s for s in submissions}

Loaded 19,984 total submissions


In [5]:
# Load all comments and group by submission
comments_by_submission = defaultdict(list)

with open(COMMENTS_PATH) as f:
    for line in f:
        comment = json.loads(line)
        # All comments in the corpus are top-level (parent_id starts with t3_)
        if comment.get("parent_id", "").startswith("t3_"):
            sub_id = comment["parent_id"][3:]
            comments_by_submission[sub_id].append(comment)

print(f"Found top-level comments for {len(comments_by_submission):,} submissions")
print(f"Total top-level comments: {sum(len(v) for v in comments_by_submission.values()):,}")

Found top-level comments for 19,984 submissions
Total top-level comments: 40,537


In [6]:
def has_real_comments(sub_id: str) -> bool:
    """Check whether a submission has at least one comment that isn't [removed] or [deleted]."""
    for c in comments_by_submission.get(sub_id, []):
        if c.get("body", "") not in ("[removed]", "[deleted]"):
            return True
    return False


def compute_flair_counts(sub_id: str) -> dict:
    """
    Compute clinician/layperson counts directly from comment flairs.

    On r/AskDocs, 'flair' is a label next to a user's name indicating whether
    they are a verified medical professional (e.g. 'Physician', 'Registered Nurse')
    or an unverified user ('Layperson/not verified as healthcare professional').

    We count from flairs rather than from the pre-computed submission field
    'layperson_top_level_comments', which has a known bug (always 0).
    """
    comments = comments_by_submission.get(sub_id, [])
    total = len(comments)
    layperson = 0
    for c in comments:
        flair = c.get("author_flair_text", "") or ""
        if "layperson" in flair.lower():
            layperson += 1
    clinician = total - layperson
    return {"total": total, "clinician": clinician, "layperson": layperson}


# Expanded term list for filtering
# Adjectives: healthy, ill, sick, unhealthy, unwell, well
# Nouns: disease, dysfunction, illness, health, disorder
TERM_PATTERN = re.compile(
    r'\b(healthy|ill|sick|unhealthy|unwell|well|disease|dysfunction|illness|health|disorder)\b',
    re.IGNORECASE
)

# Search both title AND body (selftext) for matching terms
health_submissions = [
    s for s in submissions
    if (TERM_PATTERN.search(s.get('title', '')) or TERM_PATTERN.search(s.get('selftext', '')))
    and has_real_comments(s['id'])
]

# Attach flair-based counts to each submission for downstream use
for s in health_submissions:
    counts = compute_flair_counts(s['id'])
    s['_flair_total'] = counts['total']
    s['_flair_clinician'] = counts['clinician']
    s['_flair_layperson'] = counts['layperson']

has_lay = sum(1 for s in health_submissions if s['_flair_layperson'] > 0)

print(f"OPs matching expanded term list in title or body (with >=1 real comment): {len(health_submissions):,}")
print(f"OPs with >=1 layperson comment: {has_lay} ({100*has_lay/len(health_submissions):.1f}%)")
print(f"\nSample titles:")
for s in health_submissions[:5]:
    print(f"  [{s['_flair_clinician']} clin, {s['_flair_layperson']} lay] {s['title']}")

OPs matching expanded term list in title or body (with >=1 real comment): 5,935
OPs with >=1 layperson comment: 1429 (24.1%)

Sample titles:
  [5 clin, 0 lay] How would I go about getting an amputation that is not necessary?
  [1 clin, 0 lay] How much of a calorie deficit is too much? 15AFAB
  [1 clin, 0 lay] Which medical tests to prioritize when feeling sleepy despite being healthy?
  [0 clin, 1 lay] Painful exam, is this normal
  [1 clin, 0 lay] At what point should you give up trying to figure out your illness?


In [7]:
# Select N_SAMPLE random OPs from the filtered set
sample_submissions = random.sample(health_submissions, min(N_SAMPLE, len(health_submissions)))

print(f"Selected {len(sample_submissions)} OPs for generation:")
print()

# Show comment distribution for the sample (using flair-based counts)
import statistics
sample_totals = [s['_flair_total'] for s in sample_submissions]
sample_lay = [s['_flair_layperson'] for s in sample_submissions]
sample_clin = [s['_flair_clinician'] for s in sample_submissions]

print(f"Comment distribution across {len(sample_submissions)} OPs:")
print(f"  Total comments:     mean={statistics.mean(sample_totals):.1f}, median={statistics.median(sample_totals)}, range=[{min(sample_totals)}, {max(sample_totals)}]")
print(f"  Clinician comments: mean={statistics.mean(sample_clin):.1f}, median={statistics.median(sample_clin)}")
print(f"  Layperson comments: mean={statistics.mean(sample_lay):.1f}, median={statistics.median(sample_lay)}")
has_lay = sum(1 for l in sample_lay if l > 0)
print(f"  OPs with >=1 layperson: {has_lay} ({100*has_lay/len(sample_submissions):.1f}%)")
print(f"\nEstimated API calls: DG={len(sample_submissions)}, SAG/SAGII~={sum(sample_totals)}, Total~={len(sample_submissions) + 2*sum(sample_totals)}")

Selected 100 OPs for generation:

Comment distribution across 100 OPs:
  Total comments:     mean=2.3, median=1.0, range=[1, 23]
  Clinician comments: mean=1.9, median=1.0
  Layperson comments: mean=0.4, median=0.0
  OPs with >=1 layperson: 25 (25.0%)

Estimated API calls: DG=100, SAG/SAGII~=233, Total~=566


## 3. Role Sequence Builder

For SAG and SAGII, we need to generate comments one at a time with a specified role.
We build a role sequence that matches the actual clinician/layperson counts from the
real thread and shuffle it so the order is randomized.

In [8]:
def build_role_sequence(submission: dict) -> list[str]:
    """
    Build a shuffled list of roles ('clinician' or 'layperson') matching
    the actual distribution in the real thread.

    Uses the flair-based counts computed in the filtering step:
      - _flair_clinician
      - _flair_layperson
    """
    clinician = submission.get('_flair_clinician', 0)
    layperson = submission.get('_flair_layperson', 0)

    roles = ['clinician'] * clinician + ['layperson'] * layperson
    random.shuffle(roles)
    return roles


# Quick test
test_sub = sample_submissions[0]
test_roles = build_role_sequence(test_sub)
print(f"Test OP: {test_sub['title'][:60]}")
print(f"  Total: {test_sub['_flair_total']}, Clinician: {test_sub['_flair_clinician']}, Layperson: {test_sub['_flair_layperson']}")
print(f"  Role sequence: {test_roles}")

Test OP: 17F with progressive neurological symptoms, post-op brain in
  Total: 1, Clinician: 1, Layperson: 0
  Role sequence: ['clinician']


## 4. Shared Prompt Guidelines

All three strategies share the same behavioral guidelines for how comments should
read. We define them once as a constant and inject them into each strategy's prompt.

In [9]:
COMMENT_GUIDELINES = (
    "Important guidelines for the comments:\n"
    "- Keep comments brief and natural. Match the typical length of real Reddit "
    "comments (usually 1-4 sentences).\n"
    "- Layperson comments do NOT need to be medically accurate or provide correct "
    "medical advice.\n"
    "- Each comment can focus on just one particular aspect of the OP rather than "
    "responding to the entire post.\n"
    "- Clinician comments should show empathy toward the poster.\n"
    "- Clinicians should NOT introduce themselves by stating their role or credentials "
    "(e.g. do not start with 'I'm a doctor' or 'As a nurse').\n"
    "- Clinicians should NOT include disclaimers like 'It's important to consult a "
    "physician' or 'You will want to ask a doctor about this'.\n"
    "- Laypersons should NOT disclaim their lack of medical training "
    "(e.g. do not say 'I'm not a doctor but' or 'I have no medical background').\n"
)

print("Shared guidelines defined.")
print(COMMENT_GUIDELINES)

Shared guidelines defined.
Important guidelines for the comments:
- Keep comments brief and natural. Match the typical length of real Reddit comments (usually 1-4 sentences).
- Layperson comments do NOT need to be medically accurate or provide correct medical advice.
- Each comment can focus on just one particular aspect of the OP rather than responding to the entire post.
- Clinician comments should show empathy toward the poster.
- Clinicians should NOT introduce themselves by stating their role or credentials (e.g. do not start with 'I'm a doctor' or 'As a nurse').
- Clinicians should NOT include disclaimers like 'It's important to consult a physician' or 'You will want to ask a doctor about this'.
- Laypersons should NOT disclaim their lack of medical training (e.g. do not say 'I'm not a doctor but' or 'I have no medical background').



## 5. Prompt Builders for Each Strategy

In [10]:
def build_dg_prompt(title: str, selftext: str, n_clinician: int, n_layperson: int) -> str:
    """
    DG (Discourse Generation): Single prompt that generates the entire thread.
    Uses the actual clinician/layperson counts from the real thread.
    """
    n_total = n_clinician + n_layperson

    # Build the ratio instruction
    if n_layperson == 0:
        ratio_instruction = (
            f"All {n_total} comments should imitate the response of a clinician."
        )
    elif n_clinician == 0:
        ratio_instruction = (
            f"All {n_total} comments should imitate a comment from a layperson."
        )
    else:
        ratio_instruction = (
            f"Of these comments, exactly {n_clinician} should imitate the response of a "
            f"clinician and exactly {n_layperson} should imitate a comment from a layperson."
        )

    prompt = (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        f"Generate exactly {n_total} comments (no more, no fewer) in which each comment "
        "responds to the opening post and any subsequent comments prior to it, i.e. none "
        f"of the comments should be threaded. {ratio_instruction}\n\n"
        f"{COMMENT_GUIDELINES}\n"
        f"Return your response as a JSON array of exactly {n_total} objects, where each "
        "object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\"\n\n"
        f"Opening Post:\n"
        f"Title: {title}\n\n"
        f"{selftext}"
    )
    return prompt


def build_sag_prompt(title: str, selftext: str, role: str) -> str:
    """
    SAG (Single Advice Generation): Independent prompt for a single comment.
    The model sees only the OP and the specified role.
    """
    prompt = (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        f"Generate exactly 1 comment that responds to the opening post. "
        f"The comment should imitate the response of a {role}.\n\n"
        f"{COMMENT_GUIDELINES}\n"
        "Return your response as a JSON array containing exactly 1 object with these fields:\n"
        "- \"body\": the text of the comment\n"
        f"- \"author_type\": \"{role}\"\n\n"
        f"Opening Post:\n"
        f"Title: {title}\n\n"
        f"{selftext}"
    )
    return prompt


def build_sagii_prompt(title: str, selftext: str, role: str,
                       prior_comments: list[dict]) -> str:
    """
    SAGII (SAG with Incremental Information): Prompt for a single comment that
    also includes all previously generated comments in this thread.
    """
    # Build the prior-comments section
    if prior_comments:
        prior_text = "\n\nThe following comments have already been posted in response to this OP:\n\n"
        for i, pc in enumerate(prior_comments, 1):
            prior_text += f"Comment {i} ({pc['author_type']}): {pc['body']}\n\n"
        context_instruction = (
            "Your comment should respond to the opening post. You may also acknowledge "
            "or build on the existing comments, but do not simply repeat what has already "
            "been said."
        )
    else:
        prior_text = ""
        context_instruction = "Your comment should respond to the opening post."

    prompt = (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        f"Generate exactly 1 comment. {context_instruction} "
        f"The comment should imitate the response of a {role}.\n\n"
        f"{COMMENT_GUIDELINES}\n"
        "Return your response as a JSON array containing exactly 1 object with these fields:\n"
        "- \"body\": the text of the comment\n"
        f"- \"author_type\": \"{role}\"\n\n"
        f"Opening Post:\n"
        f"Title: {title}\n\n"
        f"{selftext}"
        f"{prior_text}"
    )
    return prompt


print("Prompt builders defined (DG, SAG, SAGII).")

Prompt builders defined (DG, SAG, SAGII).


## 6. Gemini API Call Logic

In [11]:
def call_gemini(prompt: str, n_expected: int, retries: int = 3) -> list[dict]:
    """
    Send a prompt to Gemini and parse the JSON array response.
    Returns a list of comment dicts with 'body' and 'author_type' fields.
    Truncates to n_expected if the model returns more.
    """
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=1.0,
                ),
            )
            raw_text = response.text.strip()
            comments = json.loads(raw_text)

            if not isinstance(comments, list):
                raise ValueError(f"Expected a JSON array, got {type(comments).__name__}")

            validated = []
            for c in comments:
                validated.append({
                    "body": str(c.get("body", "")),
                    "author_type": str(c.get("author_type", "unknown")),
                })

            # Truncate if the model returned too many
            if len(validated) > n_expected:
                print(f"  Note: Model returned {len(validated)} comments, truncating to {n_expected}")
                validated = validated[:n_expected]

            return validated

        except json.JSONDecodeError as e:
            print(f"  JSON parse error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  API error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    print(f"  WARNING: All {retries} retries exhausted. Returning empty list.")
    return []


print("API call function defined.")

API call function defined.


## 7. Generation Functions for Each Strategy

In [12]:
def generate_dg(submission: dict) -> list[dict]:
    """
    DG strategy: one API call produces the full thread.
    Returns a list of comment dicts.
    """
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")
    clinician = submission.get('_flair_clinician', 0)
    layperson = submission.get('_flair_layperson', 0)
    total = clinician + layperson

    prompt = build_dg_prompt(title, selftext, clinician, layperson)
    comments = call_gemini(prompt, n_expected=total)
    time.sleep(DELAY_BETWEEN_CALLS)
    return comments


def generate_sag(submission: dict) -> list[dict]:
    """
    SAG strategy: n independent API calls, each generating 1 comment.
    Returns a list of comment dicts.
    """
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")
    roles = build_role_sequence(submission)

    all_comments = []
    for role in roles:
        prompt = build_sag_prompt(title, selftext, role)
        result = call_gemini(prompt, n_expected=1)
        if result:
            all_comments.append(result[0])
        else:
            all_comments.append({"body": "", "author_type": role})
        time.sleep(DELAY_BETWEEN_CALLS)

    return all_comments


def generate_sagii(submission: dict) -> list[dict]:
    """
    SAGII strategy: n sequential API calls, each seeing prior responses.
    Returns a list of comment dicts.
    """
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")
    roles = build_role_sequence(submission)

    all_comments = []
    for role in roles:
        prompt = build_sagii_prompt(title, selftext, role, prior_comments=all_comments)
        result = call_gemini(prompt, n_expected=1)
        if result:
            all_comments.append(result[0])
        else:
            all_comments.append({"body": "", "author_type": role})
        time.sleep(DELAY_BETWEEN_CALLS)

    return all_comments


print("Generation functions defined (DG, SAG, SAGII).")

Generation functions defined (DG, SAG, SAGII).


## 8. Checkpoint Utilities

In [13]:
CHECKPOINT_FILE = CHECKPOINT_DIR / "round3_progress.jsonl"


def load_checkpoint() -> tuple[dict, set]:
    """Load previously completed results. Returns (results_by_id, completed_ids)."""
    results_by_id = {}
    completed_ids = set()

    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            for line in f:
                record = json.loads(line)
                sub_id = record["submission_id"]
                results_by_id[sub_id] = record
                completed_ids.add(sub_id)
        print(f"Resumed from checkpoint: {len(completed_ids):,} OPs already completed.")
    else:
        print("No checkpoint found. Starting from scratch.")

    return results_by_id, completed_ids


def append_checkpoint(record: dict):
    """Append a single completed record to the checkpoint file."""
    with open(CHECKPOINT_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


print(f"Checkpoint file: {CHECKPOINT_FILE}")

Checkpoint file: ../output/corpora/checkpoints_round3/round3_progress.jsonl


## 9. Main Generation Loop

For each OP, we run all three strategies (DG, SAG, SAGII) and checkpoint the results.
If the notebook is interrupted and restarted, it picks up where it left off.

In [14]:
results_by_id, completed_ids = load_checkpoint()

remaining = [s for s in sample_submissions if s["id"] not in completed_ids]

print(f"Selected OPs: {len(sample_submissions)}")
print(f"Already completed: {len(sample_submissions) - len(remaining)}")
print(f"Remaining: {len(remaining)}")

No checkpoint found. Starting from scratch.
Selected OPs: 100
Already completed: 0
Remaining: 100


In [15]:
errors = []

for i, sub in enumerate(tqdm(remaining, desc="Generating (DG + SAG + SAGII)")):
    sub_id = sub["id"]
    total = sub.get('_flair_total', 0)

    if total == 0:
        continue

    try:
        # --- DG ---
        dg_comments = generate_dg(sub)

        # --- SAG ---
        sag_comments = generate_sag(sub)

        # --- SAGII ---
        sagii_comments = generate_sagii(sub)

        record = {
            "submission_id": sub_id,
            "n_comments": total,
            "n_clinician": sub.get('_flair_clinician', 0),
            "n_layperson": sub.get('_flair_layperson', 0),
            "dg": dg_comments,
            "sag": sag_comments,
            "sagii": sagii_comments,
        }

        results_by_id[sub_id] = record
        completed_ids.add(sub_id)
        append_checkpoint(record)

    except Exception as e:
        errors.append((sub_id, str(e)))
        print(f"\nError on submission {sub_id}: {e}")

print(f"\nGeneration complete. Processed {len(completed_ids):,} OPs.")
if errors:
    print(f"Errors encountered: {len(errors)}")
    for sub_id, err in errors[:10]:
        print(f"  {sub_id}: {err}")

Generating (DG + SAG + SAGII): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:48:52<00:00, 65.33s/it]


Generation complete. Processed 100 OPs.


## 10. Preview Results

In [16]:
# Preview first 3 OPs
for sub in sample_submissions[:3]:
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = comments_by_submission.get(sub_id, [])
    clinician = sub.get('_flair_clinician', 0)
    layperson = sub.get('_flair_layperson', 0)
    total = clinician + layperson

    print(f"{'='*80}")
    print(f"OP: {sub['title']}")
    print(f"Real breakdown: {clinician} clinician, {layperson} layperson ({total} total)")
    print()

    print("-- Real comments --")
    for c in sorted(real, key=lambda x: x.get("created_utc", 0)):
        if c.get("body", "") in ("[removed]", "[deleted]"):
            continue
        flair = c.get("author_flair_text") or "No flair"
        print(f"  [{flair}] {c['body']}")
        print()

    for strategy_key, strategy_label in [("dg", "DG"), ("sag", "SAG"), ("sagii", "SAGII")]:
        print(f"-- {strategy_label} --")
        for c in record.get(strategy_key, []):
            print(f"  [{c['author_type']}] {c['body']}")
            print()

    print()

OP: 17F with progressive neurological symptoms, post-op brain infection. Could this be TB brain / TBM? Risk to other kids?
Real breakdown: 1 clinician, 0 layperson (1 total)

-- Real comments --
  [Physician] A tracking sinusitis would be an unusual presentation for TB, but it should be very easy to directly test/look at the fluid removed.

The past medical history is needed. (I’m guessing HIV negative). Also, which country your sister is in is important. 

TB grows slowly and so to get that extensive disease you would need to have been unwell for longer than two weeks, typically with fever/weight loss and probably a cough over months. 

If one person around you had TB then typically many other people would be spreading and not know. It wouldn’t have to be anything to do with that particular person 3 years ago.

Direct Sinus to brain spread of infection is aggressive and lots of things can do that. Long course of antibiotics/anti-fungals anyway. 

‘Should asymptomatic children be teste

## 11. Export to Spreadsheet

Each OP gets its own tab. Within each tab:
- The Opening Post (title + body + real breakdown)
- Real Comments section
- DG section
- SAG section
- SAGII section

In [18]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
wb.remove(wb.active)  # Remove default sheet

header_font = Font(bold=True, size=11, name="Arial")
body_font = Font(size=10, name="Arial")
title_font = Font(bold=True, size=12, name="Arial")
section_fill = PatternFill('solid', fgColor='D9E1F2')  # light blue
wrap_alignment = Alignment(wrap_text=True, vertical='top')
thin_border = Border(bottom=Side(style='thin', color='CCCCCC'))

strategy_labels = {
    "dg": "Synthetic: DG (Discourse Generation)",
    "sag": "Synthetic: SAG (Single Advice Generation)",
    "sagii": "Synthetic: SAGII (SAG + Incremental Info)",
}


def sanitize_sheet_name(name: str) -> str:
    """Remove characters that Excel does not allow in sheet names."""
    for ch in ['[', ']', '*', '?', '/', '\\', ':']:
        name = name.replace(ch, '')
    return name


for idx, sub in enumerate(tqdm(sample_submissions, desc="Building spreadsheet")):
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = comments_by_submission.get(sub_id, [])
    real_sorted = sorted(real, key=lambda x: x.get("created_utc", 0))

    # Sheet name (Excel limits to 31 chars; strip illegal characters)
    short_title = sub.get("title", f"OP {idx+1}")[:26]
    sheet_name = sanitize_sheet_name(f"{idx+1}. {short_title}")
    if len(sheet_name) > 31:
        sheet_name = sheet_name[:31]
    ws = wb.create_sheet(title=sheet_name)

    ws.column_dimensions['A'].width = 18
    ws.column_dimensions['B'].width = 90

    row = 1

    # ---- Opening Post ----
    ws.cell(row=row, column=1, value="Opening Post").font = title_font
    row += 1
    ws.cell(row=row, column=1, value="Title:").font = header_font
    ws.cell(row=row, column=2, value=sub.get("title", "")).font = body_font
    ws.cell(row=row, column=2).alignment = wrap_alignment
    row += 1
    ws.cell(row=row, column=1, value="Body:").font = header_font
    ws.cell(row=row, column=2, value=sub.get("selftext", "")).font = body_font
    ws.cell(row=row, column=2).alignment = wrap_alignment
    row += 1

    clinician_count = sub.get('_flair_clinician', 0)
    layperson_count = sub.get('_flair_layperson', 0)
    total = clinician_count + layperson_count
    ws.cell(row=row, column=1, value="Real breakdown:").font = header_font
    ws.cell(row=row, column=2,
            value=f"{clinician_count} clinician, {layperson_count} layperson ({total} total)"
    ).font = body_font
    row += 2

    # ---- Real Comments ----
    ws.cell(row=row, column=1, value="Real Comments").font = title_font
    ws.cell(row=row, column=1).fill = section_fill
    ws.cell(row=row, column=2).fill = section_fill
    row += 1
    ws.cell(row=row, column=1, value="Author Type").font = header_font
    ws.cell(row=row, column=2, value="Comment").font = header_font
    row += 1
    for c in real_sorted:
        if c.get("body", "") in ("[removed]", "[deleted]"):
            continue
        flair = c.get("author_flair_text") or "No flair"
        ws.cell(row=row, column=1, value=flair).font = body_font
        ws.cell(row=row, column=1).alignment = wrap_alignment
        ws.cell(row=row, column=2, value=c.get("body", "")).font = body_font
        ws.cell(row=row, column=2).alignment = wrap_alignment
        ws.cell(row=row, column=1).border = thin_border
        ws.cell(row=row, column=2).border = thin_border
        row += 1
    row += 1

    # ---- Synthetic: DG, SAG, SAGII ----
    for strategy_key, label in strategy_labels.items():
        ws.cell(row=row, column=1, value=label).font = title_font
        ws.cell(row=row, column=1).fill = section_fill
        ws.cell(row=row, column=2).fill = section_fill
        row += 1
        ws.cell(row=row, column=1, value="Author Type").font = header_font
        ws.cell(row=row, column=2, value="Comment").font = header_font
        row += 1
        for c in record.get(strategy_key, []):
            ws.cell(row=row, column=1, value=c["author_type"]).font = body_font
            ws.cell(row=row, column=1).alignment = wrap_alignment
            ws.cell(row=row, column=2, value=c["body"]).font = body_font
            ws.cell(row=row, column=2).alignment = wrap_alignment
            ws.cell(row=row, column=1).border = thin_border
            ws.cell(row=row, column=2).border = thin_border
            row += 1
        row += 1

wb.save(OUTPUT_XLSX)
print(f"Spreadsheet saved to {OUTPUT_XLSX}")
print(f"Tabs: {wb.sheetnames}")

Building spreadsheet: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 370.56it/s]


Spreadsheet saved to ../output/corpora/round3_synthetic_comments.xlsx
Tabs: ['1. 17F with progressive neuro', '2. Hepatitis B False Positive', '3. DVT ultrasound miscommunic', '4. accidentally ate hand sani', '5. Weird bruise appearing on ', '6. New Avocado Allergy', '7. Concerning Vitamin D level', '8. Does this look like it cou', '9. What should I advocate for', '10. Slightly enlarged heart an', '11. Question about UTI, how to', '12. 32F having involuntary mus', '13. Rash on chest', '14. Possible death due to chil', '15. 17M - Stage-3A kidney dise', '16. 21 AFAB ear muscle constan', '17. Considering GLP-1 for mild', '18. feeling like im getting du', '19. What could be causing my b', '20. 26M Need help with redness', '21. Severe sudden back pain. C', '22. Please please help i have ', '23. Cat clawed me after playin', '24. My 16M mom 52F is gett', '25. Swellinglump on leg, incr', '26. “Trigeminal Neuralgia” Mis', '27. Birth control for Fibroid', '28. Lithium side effects early', '29. W

## 12. Summary Statistics

In [19]:
# Summary across all generated OPs
total_dg = 0
total_sag = 0
total_sagii = 0
total_real = 0

for sub in sample_submissions:
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = [c for c in comments_by_submission.get(sub_id, [])
            if c.get("body", "") not in ("[removed]", "[deleted]")]

    total_real += len(real)
    total_dg += len(record.get("dg", []))
    total_sag += len(record.get("sag", []))
    total_sagii += len(record.get("sagii", []))

print(f"OPs processed: {len(completed_ids)}")
print(f"Total real comments: {total_real}")
print(f"Total DG comments: {total_dg}")
print(f"Total SAG comments: {total_sag}")
print(f"Total SAGII comments: {total_sagii}")
print(f"\nAvg comments per OP:")
n = len(completed_ids) or 1
print(f"  Real: {total_real/n:.1f}")
print(f"  DG:   {total_dg/n:.1f}")
print(f"  SAG:  {total_sag/n:.1f}")
print(f"  SAGII: {total_sagii/n:.1f}")

OPs processed: 100
Total real comments: 147
Total DG comments: 233
Total SAG comments: 233
Total SAGII comments: 233

Avg comments per OP:
  Real: 1.5
  DG:   2.3
  SAG:  2.3
  SAGII: 2.3
